# Init

In [1]:
from workspace import Workspace
from util import create_recipes

# workspace
workspace = Workspace(config_path=["config/base.j2", "config/layout.j2"])
core = workspace.components["core"]

❌ core connection failed @ 192.168.137.100
🔵 core simulation api enabled
✅ printer connected @ 192.168.137.102
❌ pipettor connection failed @ /dev/ttyUSB0
[Display] socket.io connected
[Display] sending initial snapshot (145 items)
[Display] Running at 60 fps


# parameters

In [2]:
# simulation
simulation = True

# speed factor
speed_factor = 1

# tip list
tip_list = [f"{r}{c}" for r in "ABCDEFGH" for c in range(1, 13)]

# tube
tube_list = [f"{r}{c}" for r in "AB" for c in range(1,6)]
num_processed_tube = 1
falcon_rack_gravity_offset = 3

# cap
cap_list = [f"{r}{c}" for r in "CD" for c in range(1,6)]
cap_offset = [0, 0, 111-2, 0, 0, 0]
cap_gravity_offset=1

# decapper
decapper_tool_tcp_z_offset=-1

# pipette
pipetting = True
shake_travel = 7
vol = 400 #ul
immerse_depth = 20
tool_rack_1_joint = [-34.628906, 46.625977, -78.486328, 1.010742, -57.854004, -32.717285, 263.0625, 0]

# printer
dry_run_count = 1
printer_gravity_offset = 4

# inspection
inspection_frq = 4
inspection_rot = 90

# tool_rack_0
tool_rack_0_joint = [-14.39209, 40.297852, -94.614258, -0.219727, -35.024414, -13.688965, 121.89375, 0]

# main loop

In [4]:
# start
workspace.rt.start() 

# recepies
rcp = create_recipes(workspace, core, speed_factor=speed_factor)

# simulation
if not simulation:
    core.simulation(False)
    workspace.components["pipettor"].simulation(False)
    workspace.components["printer"].simulation(False)

for tip_index in tip_list:
    if pipetting:
        # pick pipettor
        rcp["tool_rack_1"].pick()
        core.robot_api.jmove(joint=tool_rack_1_joint, 
                            vel=rcp["falcon_rack"].jmove_vaj[0]*rcp["falcon_rack"].speed_factor, 
                            accel=rcp["falcon_rack"].jmove_vaj[1]*rcp["falcon_rack"].speed_factor, 
                            jerk=rcp["falcon_rack"].jmove_vaj[2]*rcp["falcon_rack"].speed_factor)

        # pick tip
        rcp["tip_rack"].pick_tip(tip_index)

        # pipette all the tubes
        for index in [num_processed_tube-1, len(tube_list)-num_processed_tube]:
            # source and destination
            aspirate_index = tube_list[index]
            dispense_index = tube_list[len(tube_list)-(index+1)]

            # immerse source and asspirate and retract
            rcp["falcon_pipepette"].immerse(anchor=aspirate_index, depth=immerse_depth)
            rcp["falcon_pipepette"].aspirate(vol=vol)
            rcp["falcon_pipepette"].retract(anchor=aspirate_index)

            # immerse target and dispense and retract
            rcp["falcon_pipepette"].immerse(anchor=dispense_index, depth=immerse_depth)
            rcp["falcon_pipepette"].dispense(vol=vol)
            rcp["falcon_pipepette"].retract(anchor=dispense_index)

        # eject tip
        rcp["waste_bin"].eject_tip(shake_travel=shake_travel)

        # place pipettor
        rcp["tool_rack_1"].place()

    # pick gripper
    rcp["tool_rack_0"].pick()
    core.robot_api.jmove(joint=tool_rack_0_joint, 
                        vel=rcp["falcon_rack"].jmove_vaj[0]*rcp["falcon_rack"].speed_factor, 
                        accel=rcp["falcon_rack"].jmove_vaj[1]*rcp["falcon_rack"].speed_factor, 
                        jerk=rcp["falcon_rack"].jmove_vaj[2]*rcp["falcon_rack"].speed_factor)
 

    # loop over all tubes, cap, print and inspect
    for index in range(num_processed_tube):
        # define the index
        tube_index = tube_list[index]
        cap_index = cap_list[index]
        
        # pick tube
        rcp["falcon_rack"].pick_from(tube_index)

        # put in decapper
        rcp["decapper"].place()

        # pick cap
        rcp["falcon_rack"].pick_from(cap_index)

        # capping and pick tube
        rcp["decapper"].cap(exit=False)
        rcp["decapper"].pick(approach=False, tool_tcp_z_offset=decapper_tool_tcp_z_offset)

        # print
        rcp["printer"].place(exit=False, gravity_offset=printer_gravity_offset)
        rcp["printer"].dry_run_spin(count=dry_run_count)
        rcp["printer"].pick(approach=False)

        # inspection present and rotate
        rcp["inspector"].present(approach=False)
        for i in range(inspection_frq):
            rcp["inspector"].rotate(rotation=inspection_rot)

        # place tube
        rcp["falcon_rack"].place_in(tube_index, gravity_offset=falcon_rack_gravity_offset, soft_approach=True)

    # decap all the tubes 
    for index in range(num_processed_tube):
        # define the index
        tube_index = tube_list[index]
        cap_index = cap_list[index]

        # pick tube
        rcp["falcon_rack"].pick_from(tube_index)  

        # put in decapper, decap
        rcp["decapper"].place(exit=False)
        rcp["decapper"].decap(approach=False)

        # place cap
        rcp["falcon_rack"].place_in(cap_index, offset=cap_offset, soft_approach=True, gravity_offset=cap_gravity_offset)

        # pick tube
        rcp["decapper"].pick(tool_tcp_z_offset=decapper_tool_tcp_z_offset)

        # place tube
        rcp["falcon_rack"].place_in(tube_index, gravity_offset=falcon_rack_gravity_offset, soft_approach=True)
    
    # place gripper
    core.robot_api.jmove(joint=tool_rack_0_joint, 
                        vel=rcp["falcon_rack"].jmove_vaj[0]*rcp["falcon_rack"].speed_factor, 
                        accel=rcp["falcon_rack"].jmove_vaj[1]*rcp["falcon_rack"].speed_factor, 
                        jerk=rcp["falcon_rack"].jmove_vaj[2]*rcp["falcon_rack"].speed_factor)
    rcp["tool_rack_0"].place()

KeyboardInterrupt: 